# HMM 择时策略 · 美股版（S&P 500）

复现框架：华西证券《HMM 模型择时及配置策略》（2021-09-09）第 2 章，思路与 A 股版 notebook 一致，标的换成 **S&P 500 指数（^GSPC）**。

**流程**（研报 2.3）：持仓期收益 / 夏普 / 成交量比等构造观测变量 → 高斯 HMM（Baum-Welch，多次随机初始化取最优似然）→ Viterbi 解码隐藏状态 → 用状态转移概率预测下一持仓期收益 → 预测 > 0 持有指数，否则空仓。

**和 A 股版相比的改动**

| 项目 | A 股版 | 本版 | 原因 |
|---|---|---|---|
| 数据 | AkShare 沪深 300 | yfinance ^GSPC（2000-01 至今） | 美股指数；本地 CSV 可替代 |
| 当前状态 | 训练集最后一个观测（上一期）的状态 | **当前调仓日观测**的状态 | 调仓日当天的特征已可得，用上一期状态等于白白滞后一期（`PREDICT_MODE="legacy"` 可切回旧口径对比） |
| 收益预测 | `A[c] @ 各状态下一期收益均值` | `A[c] @ 各状态当期收益(ret_h)均值` | 先用 A 推下一期状态分布，再取该状态的期望收益，和研报 2.3.2 表述一致 |
| 手续费 | 0 | 单边 5bp（按换手） | 更贴近实盘，`FEE=0` 可关 |
| 选参 | 全样本回测表里挑最好的 | 额外做 **验证期选参 → 测试期检验**（第 7 节） | 研报 2.5 自己也建议这么做；避免“在回测结果上挑参数” |
| 速度 | 单进程，逐期串行 | **joblib 多进程 + 结果缓存** | 各调仓日相互独立可并行；中断后重跑直接读缓存 |

**运行建议**：先把 `QUICK = True` 跑通（几分钟），确认数据和流程正常，再改成 `False` 跑完整版。完整版在 8 核机器上大约 15–30 分钟（取决于 CPU），打开 `RUN_FULL_GRID` 还会再多约 1 小时。所有结果都有缓存，中途停掉再跑会从断点继续。

研报第 3 章（多资产配置）本 notebook 暂不包含。

In [ ]:
# 如本地已安装依赖，可跳过本单元。
%pip install -q yfinance hmmlearn scikit-learn matplotlib scipy joblib

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

import hashlib
import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from joblib import Parallel, delayed
from scipy.stats import chi2, spearmanr
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
# 中文字体（Mac 自带 PingFang / Arial Unicode；Windows 用 SimHei）
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Heiti SC", "SimHei",
                                   "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. 参数区

参数选择说明（本版自定）：

- **区间 2000-01 至今**：初始 2000 个交易日训练（约 2000–2007），第一个样本外预测约在 2007 年底，回测覆盖 2008 金融危机、2020 疫情、2022 加息熊市，比研报 16 年略长。
- **基准参数沿用研报**：持仓 10 天、6 个隐藏状态、4 号观测变量组合、不定长滚动。第 5、6 节按研报做法“固定其他、每次只改一个参数”，不在中途挑最优再往下传，避免 A 股版那种“最优组合其实没被一起测过”的问题。
- **验证 / 测试切分**：2016-12-31 之前的预测用于选参，之后的约 9 年作为测试期，只看一次。

In [ ]:
@dataclass
class Config:
    symbol: str = "^GSPC"
    start_date: str = "2000-01-01"
    end_date: str | None = None        # None = 到最新交易日
    initial_train_days: int = 2000     # 研报：初始 2000 个交易日训练
    n_restarts: int = 3                # EM 随机初始化次数，取训练似然最高者
    n_iter: int = 100
    tol: float = 1e-3
    random_state: int = 42
    fee: float = 0.0005                # 单边手续费（按换手计），0 = 不计
    predict_mode: str = "transition"   # "transition"（本版）或 "legacy"（A 股版旧口径）

QUICK = True           # ← 先 True 跑通，再改 False 跑完整版
RUN_FULL_GRID = False  # True：第 7 节在完整网格上选参（更严谨，额外约 1 小时）
N_JOBS = -1            # 并行进程数，-1 = 全部 CPU 核

DATA_SOURCE = "yfinance"           # "yfinance" 或 "local_csv"
LOCAL_CSV = None                   # 本地 CSV 路径（需含 date/close/volume 列）
VALID_END = "2016-12-31"           # 第 7 节：此日期及之前的预测用于选参，之后为测试期

cfg = Config()
if QUICK:
    cfg.n_restarts, cfg.n_iter = 1, 50

FEATURE_COMBOS = {
    1: ["ret_h"],
    2: ["ret_h", "sharpe_h"],
    3: ["ret_h", "sharpe_h", "vol_ratio"],
    4: ["ret_h", "sharpe_h", "vol_ratio", "ret_half"],
}
BASE_H, BASE_N, BASE_COMBO, BASE_MODE = 10, 6, 4, "expanding"

HOLDING_GRID = [10, 20, 30, 40, 50, 60]
STATE_GRID = [3, 4, 5, 6, 7, 8]
if QUICK:
    HOLDING_GRID = [10, 20, 40]
    STATE_GRID = [4, 6, 8]

CACHE_DIR = Path("cache_us"); CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("outputs_us"); OUT_DIR.mkdir(exist_ok=True)
print(asdict(cfg)); print("QUICK =", QUICK, "| RUN_FULL_GRID =", RUN_FULL_GRID)

## 2. 读取 S&P 500 日线

默认用 yfinance 下载 `^GSPC`（价格指数，不含分红，和沪深 300 价格指数口径一致），下载后缓存到 `data_us/`，之后重跑不再联网。
如果网络不通，可以手动导出 CSV，设置 `DATA_SOURCE = "local_csv"`。

In [ ]:
def load_daily(cfg: Config) -> pd.DataFrame:
    if DATA_SOURCE == "local_csv":
        raw = pd.read_csv(LOCAL_CSV)
        raw.columns = [c.lower() for c in raw.columns]
    else:
        cache = Path("data_us") / f"{cfg.symbol.strip('^')}_{cfg.start_date}.csv"
        cache.parent.mkdir(exist_ok=True)
        fresh = cache.exists() and (time.time() - cache.stat().st_mtime) < 86400
        if fresh:
            raw = pd.read_csv(cache)
        else:
            import yfinance as yf
            d = yf.download(cfg.symbol, start=cfg.start_date, end=cfg.end_date,
                            auto_adjust=False, progress=False)
            if d.empty:
                raise RuntimeError("yfinance 未返回数据：检查网络，或改用 local_csv")
            if isinstance(d.columns, pd.MultiIndex):
                d.columns = d.columns.get_level_values(0)
            raw = d.reset_index()
            raw.columns = [str(c).lower() for c in raw.columns]
            raw.to_csv(cache, index=False)
    df = raw.rename(columns={"adj close": "adj_close"})
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df[["date", "close", "volume"]].dropna()
    df = df[(df["close"] > 0) & (df["volume"] > 0)]
    df = df[df["date"] >= pd.Timestamp(cfg.start_date)]
    if cfg.end_date:
        df = df[df["date"] <= pd.Timestamp(cfg.end_date)]
    return df.sort_values("date").reset_index(drop=True)


daily = load_daily(cfg)
print(daily["date"].min().date(), "→", daily["date"].max().date(), daily.shape)
daily.tail()

## 3. 观测变量（研报 2.4）

对每个持仓周期 `h`，每隔 `h` 个交易日取一个调仓点 `t`，构造：

- `ret_h`：过去 `h` 天收益率（价格类）
- `sharpe_h`：过去 `h` 天日收益的夏普近似（综合类）
- `vol_ratio`：过去 `h/2` 天平均成交量 / 过去 `h` 天平均成交量 − 1（成交量类）
- `ret_half`：过去 `h/2` 天收益率（价格类）

`future_ret` 是 `t → t+h` 的真实收益，只用于回测结算，**不进入任何一步训练或预测**。

In [ ]:
def build_period_observations(daily: pd.DataFrame, h: int) -> pd.DataFrame:
    close = daily["close"].to_numpy(float)
    vol = daily["volume"].to_numpy(float)
    dates = daily["date"].to_numpy()
    r = np.r_[np.nan, close[1:] / close[:-1] - 1]
    half = max(1, h // 2)
    rows = []
    for t in range(h, len(close), h):
        w = r[t - h + 1:t + 1]
        sd = np.nanstd(w, ddof=1)
        has_next = t + h < len(close)
        rows.append({
            "date": dates[t],
            "next_date": dates[t + h] if has_next else pd.NaT,
            "ret_h": close[t] / close[t - h] - 1,
            "ret_half": close[t] / close[t - half] - 1,
            "sharpe_h": np.nanmean(w) / sd * np.sqrt(h) if sd > 0 else 0.0,
            "vol_ratio": vol[t - half + 1:t + 1].mean() / vol[t - h + 1:t + 1].mean() - 1,
            "future_ret": close[t + h] / close[t] - 1 if has_next else np.nan,
            "close": close[t],
        })
    obs = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    obs = obs.dropna(subset=["ret_h", "ret_half", "sharpe_h", "vol_ratio"])
    # 最后一行 future_ret 为空：它是“今天”的信号，回测时丢弃，实盘时就是当前持仓建议
    return obs.reset_index(drop=True)


obs10 = build_period_observations(daily, 10)
print(obs10.shape)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["ret_h", "ret_half", "vol_ratio", "sharpe_h"]):
    ax.hist(obs10[col], bins=60, color="steelblue", alpha=0.8)
    ax.set_title(f"{col} (h=10)")
plt.tight_layout(); plt.show()

## 4. HMM 择时核心函数

每个调仓日 `i`：

1. 取训练窗口（不定长：从头到 `i`；定长：最近 `2000/h` 个观测），**标准化器只在窗口内 fit**；
2. 训练高斯 HMM（多次随机初始化取最优似然），Viterbi 解码窗口内的状态序列；
3. 当前状态 `c` = 调仓日观测的状态；
4. 预测收益 = `A[c] @ μ`，`μ[j]` 是窗口内状态 `j` 的平均当期收益 `ret_h`；
5. 预测 > 0 → 下一持仓期持有，否则空仓。

各调仓日互不依赖，所以用 joblib 并行；每组参数的结果缓存到 `cache_us/`，改了参数才会重算。

In [ ]:
def fit_best_hmm(X: np.ndarray, n_states: int, cfg: Config):
    logging.getLogger("hmmlearn").setLevel(logging.CRITICAL)   # 并行子进程里也静音收敛提示
    best, best_score = None, -np.inf
    for k in range(cfg.n_restarts):
        m = GaussianHMM(n_components=n_states, covariance_type="diag", n_iter=cfg.n_iter,
                        tol=cfg.tol, random_state=cfg.random_state + k, min_covar=1e-4)
        try:
            m.fit(X)
            s = m.score(X)
        except Exception:
            continue
        if np.isfinite(s) and s > best_score:
            best, best_score = m, s
    return best, best_score


def _predict_one(i, F, ret_h, fut, n_states, mode, window, cfg):
    '''单个调仓日 i 的预测。F: 全部观测的特征矩阵（numpy）。'''
    if cfg.predict_mode == "transition":
        lo = 0 if mode == "expanding" else max(0, i - window + 1)
        idx = slice(lo, i + 1)                    # 含当前观测（特征当天已知）
        target = ret_h[idx]                       # 各状态的当期收益
    else:                                         # legacy：A 股版口径
        lo = 0 if mode == "expanding" else max(0, i - window)
        idx = slice(lo, i)                        # 不含当前观测
        target = fut[idx]                         # 各状态的下一期收益（均已实现）
    scaler = StandardScaler().fit(F[idx])
    X = scaler.transform(F[idx])
    model, score = fit_best_hmm(X, n_states, cfg)
    if model is None:
        return i, np.nan, -1
    states = model.predict(X)
    overall = np.nanmean(target)
    mu = np.array([np.nanmean(target[states == s]) if (states == s).any() else overall
                   for s in range(n_states)])
    mu = np.where(np.isfinite(mu), mu, overall)
    c = int(states[-1])
    return i, float(model.transmat_[c] @ mu), c


def _cache_key(h, n_states, combo_id, mode, cfg, n_obs, last_date):
    d = dict(asdict(cfg), h=h, n=n_states, combo=combo_id, mode=mode, n_obs=n_obs, last=str(last_date))
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]


def run_hmm_timing(daily, h, n_states, combo_id, mode, cfg, verbose=True) -> pd.DataFrame:
    obs = build_period_observations(daily, h)
    key = _cache_key(h, n_states, combo_id, mode, cfg, len(obs), obs["date"].iloc[-1])
    path = CACHE_DIR / f"h{h}_n{n_states}_c{combo_id}_{mode}_{key}.pkl"
    if path.exists():
        return pd.read_pickle(path)

    window = max(30, cfg.initial_train_days // h)
    F = obs[FEATURE_COMBOS[combo_id]].to_numpy(float)
    ret_h = obs["ret_h"].to_numpy(float)
    fut = obs["future_ret"].to_numpy(float)
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS)(
        delayed(_predict_one)(i, F, ret_h, fut, n_states, mode, window, cfg)
        for i in range(window, len(obs))
    )
    pred = obs.iloc[window:][["date", "next_date", "close", "ret_h", "future_ret"]].copy()
    pred["expected_ret"] = [r[1] for r in res]
    pred["current_state"] = [r[2] for r in res]
    pred["position"] = (pred["expected_ret"] > 0).astype(int)
    pred = pred.assign(h=h, n_states=n_states, combo_id=combo_id, mode=mode).reset_index(drop=True)
    pred.to_pickle(path)
    if verbose:
        print(f"  h={h:<3} n={n_states} combo={combo_id} {mode:<9} "
              f"{len(res)} 次训练, {time.time() - t0:5.1f}s")
    return pred

In [ ]:
def backtest(pred: pd.DataFrame, fee: float) -> pd.DataFrame:
    bt = pred.dropna(subset=["future_ret"]).copy()      # 最后一期（今天的信号）不结算
    bt["turnover"] = bt["position"].diff().abs().fillna(bt["position"].abs())
    bt["strategy_ret"] = bt["position"] * bt["future_ret"] - fee * bt["turnover"]
    bt["benchmark_ret"] = bt["future_ret"]
    bt["strategy_nav"] = (1 + bt["strategy_ret"]).cumprod()
    bt["benchmark_nav"] = (1 + bt["benchmark_ret"]).cumprod()
    return bt


def max_drawdown(nav: pd.Series) -> float:
    nav = pd.concat([pd.Series([1.0]), nav.reset_index(drop=True)])
    return float(abs((nav / nav.cummax() - 1).min()))


def metrics(bt: pd.DataFrame, start=None, end=None) -> dict:
    '''在 [start, end] 区间（按调仓日）重新计算指标；净值从 1 重新起算。'''
    b = bt
    if start is not None: b = b[b["date"] > pd.Timestamp(start)]
    if end is not None:   b = b[b["date"] <= pd.Timestamp(end)]
    if len(b) < 3:
        return {}
    years = (b["next_date"].iloc[-1] - b["date"].iloc[0]).days / 365.25
    ppy = len(b) / years
    out = {}
    for tag, col in [("", "strategy_ret"), ("bm_", "benchmark_ret")]:
        r = b[col]
        nav = (1 + r).cumprod()
        total = nav.iloc[-1] - 1
        out[tag + "sharpe"] = r.mean() / r.std() * np.sqrt(ppy) if r.std() > 0 else np.nan
        out[tag + "annual_return"] = (1 + total) ** (1 / years) - 1
        out[tag + "total_return"] = total
        out[tag + "max_drawdown"] = max_drawdown(nav)
    held = b[b["position"] == 1]
    out["win_rate"] = float((np.sign(b["expected_ret"]) == np.sign(b["future_ret"])).mean())  # 方向判断正确率（A 股版口径）
    out["hold_win_rate"] = float((held["future_ret"] > 0).mean()) if len(held) else np.nan    # 持仓期里赚钱的比例
    out["exposure"] = float(b["position"].mean())
    out["trade_count"] = int((b["position"].diff().abs() > 0).sum() + b["position"].iloc[0])
    out["n_periods"] = len(b)
    out["start"], out["end"] = b["date"].iloc[0].date(), b["next_date"].iloc[-1].date()
    return out


CN = {"sharpe": "夏普", "annual_return": "年化收益", "total_return": "总收益", "max_drawdown": "最大回撤",
      "win_rate": "方向胜率", "hold_win_rate": "持仓胜率", "exposure": "持仓占比", "trade_count": "交易次数"}


def table(rows: list[dict], key: str) -> pd.DataFrame:
    '''研报样式：每行一个参数，末尾附基准。'''
    df = pd.DataFrame(rows)
    strat = df[[key] + list(CN)].rename(columns=CN)
    bm = df.iloc[[0]][["bm_sharpe", "bm_annual_return", "bm_total_return", "bm_max_drawdown"]]
    bm.columns = ["夏普", "年化收益", "总收益", "最大回撤"]
    bm.insert(0, key, "基准(S&P500)")
    return pd.concat([strat, bm], ignore_index=True)


def run_case(h, n, combo, mode):
    pred = run_hmm_timing(daily, h, n, combo, mode, cfg)
    bt = backtest(pred, cfg.fee)
    return pred, bt, metrics(bt)

## 5. 隐藏状态的马尔可夫性检验（研报 2.4.1）

和研报一样，用最初 2000 个交易日（约 2000–2007）作为训练样本，训练 HMM 后 Viterbi 解码，对状态序列做卡方检验：

```text
chi2 = 2 * Σ_ij f_ij * log(p_ij / p_.j),  自由度 (m-1)^2
```

统计量大于 0.05 显著性下的临界值，则认为状态序列具有马氏性。`strength = 统计量 / 临界值`，越大越明显。

In [ ]:
markov_daily = daily.iloc[:cfg.initial_train_days].reset_index(drop=True)
print("2.4.1 训练样本:", markov_daily["date"].min().date(), "→", markov_daily["date"].max().date())


def decode_in_sample(daily_sample, h, n_states, combo_id, cfg):
    obs = build_period_observations(daily_sample, h)
    X = StandardScaler().fit_transform(obs[FEATURE_COMBOS[combo_id]].to_numpy(float))
    model, score = fit_best_hmm(X, n_states, cfg)
    obs = obs.copy()
    obs["state"] = model.predict(X)
    return obs, model


def markov_test(states, m, alpha=0.05):
    f = np.zeros((m, m))
    for a, b in zip(states[:-1], states[1:]):
        f[a, b] += 1
    p_ij = np.divide(f, f.sum(1, keepdims=True), out=np.zeros_like(f), where=f.sum(1, keepdims=True) > 0)
    p_j = np.broadcast_to(f.sum(0, keepdims=True) / f.sum(), f.shape)
    mask = (f > 0) & (p_ij > 0) & (p_j > 0)
    stat = 2 * np.sum(f[mask] * np.log(p_ij[mask] / p_j[mask]))
    crit = chi2.ppf(1 - alpha, (m - 1) ** 2)
    return {"临界值": crit, "统计量": stat, "strength": stat / crit, "通过": stat > crit}


rows = []
for n in [3, 4, 5, 6, 7, 8]:
    obs_m, _ = decode_in_sample(markov_daily, BASE_H, n, BASE_COMBO, cfg)
    rows.append({"隐藏状态数": n, **markov_test(obs_m["state"].to_numpy(), n)})
markov_state_table = pd.DataFrame(rows)
print("表 1：持仓 10 天，改变隐藏状态数"); display(markov_state_table)

rows = []
for h in [5, 10, 20, 30, 40]:
    obs_m, _ = decode_in_sample(markov_daily, h, BASE_N, BASE_COMBO, cfg)
    rows.append({"持仓周期": h, "观测数": len(obs_m), **markov_test(obs_m["state"].to_numpy(), BASE_N)})
markov_holding_table = pd.DataFrame(rows)
print("表 2：6 个隐藏状态，改变持仓周期"); display(markov_holding_table)

markov_state_table.to_csv(OUT_DIR / "t1_markov_states.csv", index=False, encoding="utf-8-sig")
markov_holding_table.to_csv(OUT_DIR / "t2_markov_holding.csv", index=False, encoding="utf-8-sig")

### 5.1 隐藏状态解码效果（研报图 6–9）

左：训练样本内 Viterbi 解码，不同颜色是不同状态。右：只在某个状态出现时买入持有一期的“分时买入”净值，用来看各状态是否有稳定的涨跌含义。

注意这是**样本内**解码，只用于解释模型学到了什么，不代表样本外表现。

In [ ]:
def plot_decode(daily_sample, h, n_states, combo_id, title):
    obs_d, _ = decode_in_sample(daily_sample, h, n_states, combo_id, cfg)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [3, 2]})
    a1.plot(daily_sample["date"], daily_sample["close"], color="lightgray", lw=1.2)
    colors = plt.cm.tab10(np.arange(n_states))
    prof = []
    for s in range(n_states):
        part = obs_d[obs_d["state"] == s]
        if part.empty:
            continue
        a1.scatter(part["date"], part["close"], s=14, color=colors[s], label=f"state {s}")
        r = np.where(obs_d["state"] == s, obs_d["future_ret"].fillna(0), 0.0)
        a2.plot(obs_d["date"], np.cumprod(1 + r), color=colors[s], label=f"state {s}")
        prof.append({"state": s, "次数": len(part), "当期收益均值": part["ret_h"].mean(),
                     "下期收益均值": part["future_ret"].mean(), "下期上涨概率": (part["future_ret"] > 0).mean()})
    a1.set_title(title); a1.legend(ncol=3, fontsize=8)
    a2.axhline(1, color="k", lw=0.8); a2.set_title("各状态分时买入净值"); a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    return fig, pd.DataFrame(prof)


for n in [3, 6]:
    fig, prof = plot_decode(markov_daily, BASE_H, n, BASE_COMBO, f"S&P500 解码, h={BASE_H}, n_states={n}（样本内）")
    display(prof)
    fig.savefig(OUT_DIR / f"fig_decode_n{n}.png", dpi=150, bbox_inches="tight")

## 6. 滚动训练回测（研报 2.4.2 / 2.4.3）

完全按研报做法：以“持仓 10 天、6 个状态、4 号组合”为基准，每张表只改一个参数，不定长（expanding）和定长（fixed）各做一遍。

In [ ]:
def sensitivity(mode):
    t_all = time.time()
    out = {}
    print(f"[{mode}] 表 3：观测变量组合")
    rows, bts = [], {}
    for c in FEATURE_COMBOS:
        _, bt, m = run_case(BASE_H, BASE_N, c, mode); rows.append({"组合": c, **m}); bts[c] = bt
    out["combo"] = (table(rows, "组合"), bts)

    print(f"[{mode}] 表 4：隐藏状态数")
    rows, bts = [], {}
    for n in STATE_GRID:
        _, bt, m = run_case(BASE_H, n, BASE_COMBO, mode); rows.append({"状态数": n, **m}); bts[n] = bt
    out["state"] = (table(rows, "状态数"), bts)

    print(f"[{mode}] 表 5：持仓周期（每个周期样本起点不同，各自附基准）")
    parts, bts = [], {}
    for h in HOLDING_GRID:
        _, bt, m = run_case(h, BASE_N, BASE_COMBO, mode); bts[h] = bt
        parts.append(table([{"持仓周期": h, **m}], "持仓周期"))
    out["holding"] = (pd.concat(parts, ignore_index=True), bts)
    print(f"[{mode}] 完成，用时 {(time.time() - t_all) / 60:.1f} 分钟")
    return out


def show(res, mode):
    for k, title in [("combo", "观测变量组合"), ("state", "隐藏状态数"), ("holding", "持仓周期")]:
        tb, bts = res[k]
        print(f"\n{mode} · {title}"); display(tb)
        tb.to_csv(OUT_DIR / f"{mode}_{k}.csv", index=False, encoding="utf-8-sig")
        fig, ax = plt.subplots(figsize=(12, 4.5))
        for p, bt in bts.items():
            ax.plot(bt["date"], bt["strategy_nav"], label=f"{k}={p}")
        ref = list(bts.values())[0]
        ax.plot(ref["date"], ref["benchmark_nav"], "k--", label="S&P500")
        ax.set_title(f"{mode} · {title}"); ax.legend(ncol=4, fontsize=8)
        plt.tight_layout(); plt.show()
        fig.savefig(OUT_DIR / f"{mode}_{k}_nav.png", dpi=150, bbox_inches="tight")


res_exp = sensitivity("expanding")
show(res_exp, "expanding")

In [ ]:
res_fix = sensitivity("fixed")
show(res_fix, "fixed")

### 6.1 定长 vs 不定长（研报表 6、图 10–13）

In [ ]:
bt_e = res_exp["combo"][1][BASE_COMBO]
bt_f = res_fix["combo"][1][BASE_COMBO]
t6 = table([{"训练方式": "定长滚动", **metrics(bt_f)}, {"训练方式": "不定长滚动", **metrics(bt_e)}], "训练方式")
display(t6); t6.to_csv(OUT_DIR / "t6_fixed_vs_expanding.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for col, (bt, name) in enumerate([(bt_f, "定长滚动"), (bt_e, "不定长滚动")]):
    axes[0, col].plot(bt["date"], bt["strategy_nav"], label="HMM 择时")
    axes[0, col].plot(bt["date"], bt["benchmark_nav"], "k--", label="S&P500")
    axes[0, col].set_title(f"{name}：净值"); axes[0, col].legend()
    axes[1, col].plot(bt["date"], bt["strategy_nav"] / bt["benchmark_nav"] - 1, color="tab:red")
    axes[1, col].axhline(0, color="k", lw=0.8); axes[1, col].set_title(f"{name}：累计超额（相对净值 − 1）")
plt.tight_layout(); plt.show()
fig.savefig(OUT_DIR / "fig_fixed_vs_expanding.png", dpi=150, bbox_inches="tight")

## 7. 样本外选参检验（研报 2.5 的建议）

第 6 节的表都是在**整个回测区间**上比较参数，挑出的“最优参数”带有事后选择偏差。这里按研报 2.5 的建议拆开：

- **验证期**（首个预测日 ~ `VALID_END`）：在候选参数里按夏普选出最优；
- **测试期**（`VALID_END` 之后）：用选定参数的表现作为真正的样本外结果，只看一次。

因为每个调仓日的预测只用了当时之前的数据，第 6 节算好的预测可以直接按日期切开复用，不用重算。
`RUN_FULL_GRID = True` 时候选集是完整网格（组合 × 状态数 × 持仓周期 × 训练方式），否则只用第 6 节已跑过的参数。

In [ ]:
if RUN_FULL_GRID:
    grid = [(h, n, c, mode) for mode in ["expanding", "fixed"] for c in FEATURE_COMBOS
            for n in STATE_GRID for h in HOLDING_GRID]
    print(f"完整网格 {len(grid)} 组（已缓存的会直接读取）")
else:
    grid = sorted({(BASE_H, BASE_N, c, m) for c in FEATURE_COMBOS for m in ["expanding", "fixed"]}
                  | {(BASE_H, n, BASE_COMBO, m) for n in STATE_GRID for m in ["expanding", "fixed"]}
                  | {(h, BASE_N, BASE_COMBO, m) for h in HOLDING_GRID for m in ["expanding", "fixed"]})
    print(f"候选 {len(grid)} 组（第 6 节已跑过）")

rows = []
for h, n, c, mode in grid:
    pred = run_hmm_timing(daily, h, n, c, mode, cfg, verbose=RUN_FULL_GRID)
    bt = backtest(pred, cfg.fee)
    v, t = metrics(bt, end=VALID_END), metrics(bt, start=VALID_END)
    if not v or not t:
        continue
    rows.append({"h": h, "n_states": n, "combo": c, "mode": mode,
                 "valid_sharpe": v["sharpe"], "valid_ann": v["annual_return"], "valid_mdd": v["max_drawdown"],
                 "valid_bm_sharpe": v["bm_sharpe"],
                 "test_sharpe": t["sharpe"], "test_ann": t["annual_return"], "test_mdd": t["max_drawdown"],
                 "test_bm_sharpe": t["bm_sharpe"], "test_bm_ann": t["bm_annual_return"],
                 "test_bm_mdd": t["bm_max_drawdown"], "test_exposure": t["exposure"]})
sel = pd.DataFrame(rows).sort_values("valid_sharpe", ascending=False).reset_index(drop=True)
sel.to_csv(OUT_DIR / "t7_valid_test_grid.csv", index=False, encoding="utf-8-sig")

rho, pval = spearmanr(sel["valid_sharpe"], sel["test_sharpe"])
print(f"验证期夏普 vs 测试期夏普 的 Spearman 秩相关：{rho:.3f}（p={pval:.3f}）"
      " —— 接近 0 说明验证期的好参数到测试期不一定还好，即参数不稳定")
print("\n验证期夏普前 10 名及其测试期表现：")
display(sel.head(10))

In [ ]:
best = sel.iloc[0]
h, n, c, mode = int(best["h"]), int(best["n_states"]), int(best["combo"]), best["mode"]
print(f"验证期选出的参数：h={h}, n_states={n}, combo={c} {FEATURE_COMBOS[c]}, {mode}")

bt_best = backtest(run_hmm_timing(daily, h, n, c, mode, cfg, verbose=False), cfg.fee)
tv, tt = metrics(bt_best, end=VALID_END), metrics(bt_best, start=VALID_END)
final = pd.DataFrame([
    {"区间": "验证期", "对象": "HMM", **{CN[k]: tv[k] for k in CN}},
    {"区间": "验证期", "对象": "S&P500", "夏普": tv["bm_sharpe"], "年化收益": tv["bm_annual_return"],
     "总收益": tv["bm_total_return"], "最大回撤": tv["bm_max_drawdown"]},
    {"区间": "测试期", "对象": "HMM", **{CN[k]: tt[k] for k in CN}},
    {"区间": "测试期", "对象": "S&P500", "夏普": tt["bm_sharpe"], "年化收益": tt["bm_annual_return"],
     "总收益": tt["bm_total_return"], "最大回撤": tt["bm_max_drawdown"]},
])
print(f"验证期 {tv['start']} → {tv['end']}，测试期 {tt['start']} → {tt['end']}")
display(final); final.to_csv(OUT_DIR / "t7_final_oos.csv", index=False, encoding="utf-8-sig")

test = bt_best[bt_best["date"] > pd.Timestamp(VALID_END)]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test["date"], (1 + test["strategy_ret"]).cumprod(), label=f"HMM (h={h}, n={n}, combo={c}, {mode})")
ax.plot(test["date"], (1 + test["benchmark_ret"]).cumprod(), "k--", label="S&P500")
ax.fill_between(test["date"], 0, 1, where=test["position"] == 0, transform=ax.get_xaxis_transform(),
                color="gray", alpha=0.12, step="post", label="空仓")
ax.set_title("测试期净值（样本外，参数只在验证期选定）"); ax.legend()
plt.tight_layout(); plt.show()
fig.savefig(OUT_DIR / "fig_test_nav.png", dpi=150, bbox_inches="tight")

latest = run_hmm_timing(daily, h, n, c, mode, cfg, verbose=False).iloc[-1]
print(f"最近一个调仓点 {pd.Timestamp(latest['date']).date()}：预测下期收益 {latest['expected_ret']:+.4f}，"
      f"建议{'持有' if latest['position'] else '空仓'}（仅为模型输出，不构成投资建议）")

## 8. 结论（跑完后填写）

建议从这几个角度写：

1. **马氏性**：S&P 500 的解码状态是否通过检验，和研报（沪深 300 全部通过）对比。
2. **全样本敏感性**（第 6 节）：哪些参数跑赢基准；结果随参数变化是否平滑。如果相邻参数（例如持仓 30 天和 40 天）结果差别很大，说明策略对参数敏感。
3. **样本外**（第 7 节）：验证期选出的参数在测试期是否仍然跑赢；验证期和测试期夏普的秩相关大概是多少。
4. **和 A 股的差异**：美股长期单边上涨、空仓成本高，择时策略通常更难跑赢买入持有，但可能在回撤上有优势。可以对比 `最大回撤` 和 `持仓占比`。

局限：`^GSPC` 是价格指数，不含分红（约每年 1.5–2%），空仓期也没算现金利息，两者对基准和策略的影响方向相反；手续费按单边 5bp 估算。